# Physics-Informed ELM (PIELM) Benchmarks

**Key change from previous notebook:**

The previous notebook used **supervised ELM** — it minimized `||H*beta - u_true||^2`, requiring the analytical solution during training. This is unfair vs PINN, which only knows the PDE.

This notebook implements **PIELM** (Dwivedi & Srinivasan 2020): the PDE operator is applied analytically to the random basis `sin(w*x + b)`, and beta is found by solving the PDE + boundary conditions — **no u_true during training**.

For phi_j(x) = sin(w_j*x + b_j):
- phi_j''(x) = -w_j^2 * sin(w_j*x + b_j)

Now ELM and PINN solve the **same problem** (discover solution from physics only).
u_true is used **only for evaluation** (computing L2 error).

**Also fixed:** Weight scale is tuned per problem instead of fixed *0.1.

In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# --- Weight Initialization ---
def power_law_init(shape, alpha=2.0, w_scale=1.0):
    u = torch.rand(shape, device=device).clamp(1e-7, 1 - 1e-7)
    weights = (1 - u) ** (1 / (1 - alpha))
    weights = (weights - weights.mean()) / (weights.std() + 1e-7)
    return weights * w_scale

def gaussian_init(shape, w_scale=1.0):
    return torch.randn(shape, device=device) * w_scale

# --- PIELM Solvers ---

def solve_pielm_poisson(x, f_source, x_bc, u_bc, h_dim, init_type="power",
                         w_scale=5.0, b_scale=1.0, lambd=1e-6, bc_weight=100.0):
    """PIELM for -u""(x) = f(x) with Dirichlet BCs."""
    W = power_law_init((1, h_dim), w_scale=w_scale) if init_type == "power" \
        else gaussian_init((1, h_dim), w_scale=w_scale)
    b = torch.randn(h_dim, device=device) * b_scale

    start = time.time()
    H = torch.sin(x @ W + b)
    H_pde = (W ** 2) * H  # -u"" = f => W^2 * H @ beta = f

    H_bc = torch.sin(x_bc @ W + b)

    A = torch.cat([H_pde, bc_weight * H_bc], dim=0)
    rhs = torch.cat([f_source, bc_weight * u_bc], dim=0)

    ATA = A.t() @ A + lambd * torch.eye(h_dim, device=device)
    ATb = A.t() @ rhs
    beta = torch.linalg.solve(ATA, ATb)
    t_solve = time.time() - start

    u_pred = H @ beta
    return u_pred, beta, W, b, t_solve


def solve_pielm_helmholtz(x, f_source, k, x_bc, u_bc, h_dim, init_type="power",
                            w_scale=15.0, b_scale=1.0, lambd=1e-6, bc_weight=100.0):
    """PIELM for -u""(x) - k^2*u(x) = f(x) with Dirichlet BCs."""
    W = power_law_init((1, h_dim), w_scale=w_scale) if init_type == "power" \
        else gaussian_init((1, h_dim), w_scale=w_scale)
    b = torch.randn(h_dim, device=device) * b_scale

    start = time.time()
    H = torch.sin(x @ W + b)
    H_pde = (W ** 2 - k ** 2) * H

    H_bc = torch.sin(x_bc @ W + b)

    A = torch.cat([H_pde, bc_weight * H_bc], dim=0)
    rhs = torch.cat([f_source, bc_weight * u_bc], dim=0)

    ATA = A.t() @ A + lambd * torch.eye(h_dim, device=device)
    ATb = A.t() @ rhs
    beta = torch.linalg.solve(ATA, ATb)
    t_solve = time.time() - start

    u_pred = H @ beta
    return u_pred, beta, W, b, t_solve


def solve_pielm_advdiff(x, x_bc, u_bc, h_dim, eps, init_type="power",
                          w_scale=60.0, b_scale=1.0, lambd=1e-6, bc_weight=100.0):
    """PIELM for -eps*u""(x) + u'(x) = 0 with Dirichlet BCs."""
    W = power_law_init((1, h_dim), w_scale=w_scale) if init_type == "power" \
        else gaussian_init((1, h_dim), w_scale=w_scale)
    b = torch.randn(h_dim, device=device) * b_scale

    start = time.time()
    arg = x @ W + b
    H = torch.sin(arg)
    H_cos = torch.cos(arg)
    H_pde = eps * (W ** 2) * H + W * H_cos

    H_bc = torch.sin(x_bc @ W + b)

    A = torch.cat([H_pde, bc_weight * H_bc], dim=0)
    f_zero = torch.zeros(x.shape[0], 1, device=device)
    rhs = torch.cat([f_zero, bc_weight * u_bc], dim=0)

    ATA = A.t() @ A + lambd * torch.eye(h_dim, device=device)
    ATb = A.t() @ rhs
    beta = torch.linalg.solve(ATA, ATb)
    t_solve = time.time() - start

    u_pred = H @ beta
    return u_pred, beta, W, b, t_solve


def solve_pielm_poisson_2d(X, f_source, X_bc, u_bc, h_dim, init_type="power",
                             w_scale=10.0, b_scale=1.0, lambd=1e-6, bc_weight=100.0):
    """PIELM for -laplacian(u) = f on 2D with Dirichlet BCs."""
    W = power_law_init((2, h_dim), w_scale=w_scale) if init_type == "power" \
        else gaussian_init((2, h_dim), w_scale=w_scale)
    b = torch.randn(h_dim, device=device) * b_scale

    start = time.time()
    H = torch.sin(X @ W + b)
    W_sq_sum = (W ** 2).sum(dim=0, keepdim=True)
    H_pde = W_sq_sum * H

    H_bc = torch.sin(X_bc @ W + b)

    A = torch.cat([H_pde, bc_weight * H_bc], dim=0)
    rhs = torch.cat([f_source, bc_weight * u_bc], dim=0)

    ATA = A.t() @ A + lambd * torch.eye(h_dim, device=device)
    ATb = A.t() @ rhs
    beta = torch.linalg.solve(ATA, ATb)
    t_solve = time.time() - start

    u_pred = H @ beta
    return u_pred, beta, W, b, t_solve


print("PIELM solvers loaded: Poisson, Helmholtz, Advection-Diffusion, Poisson 2D")

# ELM Classification: Power-Law vs Gaussian vs Backprop

**MNIST and FashionMNIST** — both ELM variants and backprop receive the same (image, label) pairs.
This is a fair supervised comparison demonstrating that Power-Law initialization improves ELM accuracy.

In [ ]:
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim

def run_classification_experiment(dataset_name='MNIST'):
    print(f"Starting {dataset_name} experiment...")
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    if dataset_name == 'MNIST':
        train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else:
        train_set = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_set = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    train_loader_bp = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
    x_train = train_set.data.view(-1, 784).float() / 255.0
    y_train = torch.nn.functional.one_hot(train_set.targets, 10).float()
    x_test = test_set.data.view(-1, 784).float() / 255.0
    y_test = test_set.targets
    hidden_dim = 1024

    def run_elm(init_type, label):
        torch.cuda.synchronize() if device.type == 'cuda' else None
        start = time.time()
        W = power_law_init((784, hidden_dim), w_scale=0.1) if init_type == 'power' \
            else gaussian_init((784, hidden_dim), w_scale=0.1)
        b = torch.randn(hidden_dim, device=device) * 0.1
        H = torch.sigmoid(torch.matmul(x_train.to(device), W) + b)
        I = torch.eye(hidden_dim, device=device)
        C = 1e-2
        HTH = torch.matmul(H.t(), H)
        HTY = torch.matmul(H.t(), y_train.to(device))
        beta = torch.matmul(torch.inverse(HTH + I/C), HTY)
        torch.cuda.synchronize() if device.type == 'cuda' else None
        elapsed = time.time() - start
        h_test = torch.sigmoid(torch.matmul(x_test.to(device), W) + b)
        y_pred = torch.matmul(h_test, beta)
        acc = (y_pred.argmax(1) == y_test.to(device)).float().mean().item()
        return acc, elapsed

    # ELM Power Law
    pl_acc, pl_time = run_elm('power', 'Power Law')
    # ELM Gaussian
    g_acc, g_time = run_elm('gaussian', 'Gaussian')

    # Backprop
    model = nn.Sequential(nn.Linear(784, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 10)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    torch.cuda.synchronize() if device.type == 'cuda' else None
    start_bp = time.time()
    for epoch in range(5):
        for imgs, lbls in train_loader_bp:
            imgs = imgs.view(-1, 784).to(device)
            lbls = lbls.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()
    torch.cuda.synchronize() if device.type == 'cuda' else None
    bp_time = time.time() - start_bp

    model.eval()
    with torch.no_grad():
        bp_outputs = model(x_test.to(device))
        bp_acc = (bp_outputs.argmax(1) == y_test.to(device)).float().mean().item()

    return {
        "Dataset": dataset_name,
        "ELM (Power Law) Acc": pl_acc * 100, "ELM (PL) Time": pl_time,
        "ELM (Gaussian) Acc": g_acc * 100, "ELM (G) Time": g_time,
        "Backprop Acc": bp_acc * 100, "BP Time": bp_time,
    }

results_list = [run_classification_experiment('MNIST'), run_classification_experiment('FashionMNIST')]
df_class = pd.DataFrame(results_list)
print("\nClassification Results:")
print(df_class.to_markdown(index=False, floatfmt=".2f"))

# --- Plot ---
from matplotlib.lines import Line2D
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
plt.rcParams.update({'font.size': 12, 'font.family': 'serif'})

sample_W = power_law_init((2048, 2048), alpha=2.0, w_scale=0.1).cpu().numpy().flatten()
ax1.hist(sample_W, bins=100, color='#6a0dad', alpha=0.7, density=True, edgecolor='black', linewidth=0.5)
ax1.set_yscale('log')
ax1.set_title(r'(A) Weight Distribution ($P(w) \sim w^{-2.0}$)', fontsize=14)
ax1.set_xlabel('Weight Value (w)'); ax1.set_ylabel('Density (Log Scale)')
ax1.grid(True, which='major', linestyle='--', linewidth=0.5, alpha=0.7)

colors = {'PL': '#1f77b4', 'G': '#ff7f0e', 'BP': '#d62728'}
markers = {'MNIST': 'o', 'FashionMNIST': 's'}
for res in results_list:
    ds = res['Dataset']
    ax2.scatter(res['ELM (PL) Time'], res['ELM (Power Law) Acc'], color=colors['PL'], marker=markers[ds], s=180, edgecolor='black')
    ax2.scatter(res['ELM (G) Time'], res['ELM (Gaussian) Acc'], color=colors['G'], marker=markers[ds], s=180, edgecolor='black')
    ax2.scatter(res['BP Time'], res['Backprop Acc'], color=colors['BP'], marker=markers[ds], s=180, edgecolor='black')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', label='MNIST', markersize=12, markeredgecolor='black'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', label='Fashion-MNIST', markersize=12, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['PL'], label='ELM (Power Law)', markersize=12, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['G'], label='ELM (Gaussian)', markersize=12, markeredgecolor='black'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['BP'], label='Backprop (Adam)', markersize=12, markeredgecolor='black'),
]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax2.set_xscale('log')
ax2.set_title('(B) Speed vs Accuracy', fontsize=14)
ax2.set_xlabel('Training Time (s) - Log Scale'); ax2.set_ylabel('Test Accuracy (%)')
ax2.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.7)

plt.tight_layout()
plt.savefig('elm_vs_bp_results.png', dpi=300, bbox_inches='tight')
print("Plot saved as elm_vs_bp_results.png")
plt.show()

# Regularization Sweep: Power-Law vs Gaussian

Logarithmic sweep over lambda to find optimal regularization and compare the U-curve shape for both initialization methods.

In [ ]:
lambdas = np.logspace(-10, 2, 25)
N_SEEDS_SWEEP = 10
h_dim_sweep = 4096
W_SCALE_SWEEP = 5.0  # tuned scale for Poisson

x_sweep = torch.linspace(-1, 1, 30000, device=device).view(-1, 1)
f_sweep = (np.pi ** 2) * torch.sin(np.pi * x_sweep)
u_true_sweep = torch.sin(np.pi * x_sweep)
x_bc_sweep = torch.tensor([[-1.0], [1.0]], device=device)
u_bc_sweep = torch.zeros(2, 1, device=device)

results_pl = {lam: [] for lam in lambdas}
results_g = {lam: [] for lam in lambdas}

print(f"Lambda sweep ({N_SEEDS_SWEEP} seeds x {len(lambdas)} lambdas x 2 inits)...")
for seed in range(N_SEEDS_SWEEP):
    for init_type, results_dict in [('power', results_pl), ('gaussian', results_g)]:
        torch.manual_seed(seed)
        W = power_law_init((1, h_dim_sweep), w_scale=W_SCALE_SWEEP) if init_type == 'power' \
            else gaussian_init((1, h_dim_sweep), w_scale=W_SCALE_SWEEP)
        b = torch.randn(h_dim_sweep, device=device) * 1.0

        H = torch.sin(x_sweep @ W + b)
        H_pde = (W ** 2) * H
        H_bc = torch.sin(x_bc_sweep @ W + b)
        bc_weight = 100.0

        for lam in lambdas:
            A = torch.cat([H_pde, bc_weight * H_bc], dim=0)
            rhs = torch.cat([f_sweep, bc_weight * u_bc_sweep], dim=0)
            ATA = A.t() @ A + lam * torch.eye(h_dim_sweep, device=device)
            ATb = A.t() @ rhs
            beta = torch.linalg.solve(ATA, ATb)
            u_pred = H @ beta
            err = (torch.norm(u_true_sweep - u_pred) / torch.norm(u_true_sweep)).item()
            results_dict[lam].append(err)

means_pl = [np.mean(results_pl[l]) for l in lambdas]
means_g = [np.mean(results_g[l]) for l in lambdas]
stds_pl = [np.std(results_pl[l]) for l in lambdas]
stds_g = [np.std(results_g[l]) for l in lambdas]

best_lam_pl = lambdas[np.argmin(means_pl)]
best_lam_g = lambdas[np.argmin(means_g)]
print(f"Best lambda (Power Law): {best_lam_pl:.2e} (err={min(means_pl):.6f})")
print(f"Best lambda (Gaussian):  {best_lam_g:.2e} (err={min(means_g):.6f})")

fig, ax = plt.subplots(figsize=(10, 6))
plt.rcParams.update({'font.size': 12, 'font.family': 'serif'})

ax.errorbar(lambdas, means_pl, yerr=stds_pl, fmt='o-', color='#2ca02c', capsize=4, lw=2, ms=7, label='Power Law')
ax.errorbar(lambdas, means_g, yerr=stds_g, fmt='s--', color='#ff7f0e', capsize=4, lw=2, ms=7, label='Gaussian')

ax.axvline(best_lam_pl, color='#2ca02c', linestyle=':', alpha=0.5)
ax.axvline(best_lam_g, color='#ff7f0e', linestyle=':', alpha=0.5)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Regularisation lambda', fontsize=13)
ax.set_ylabel('L2 Relative Error (PIELM)', fontsize=13)
ax.set_title('Lambda Sweep: Power-Law vs Gaussian (PIELM Poisson)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, which='major', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('lambda_sweep_pielm.png', dpi=300, bbox_inches='tight')
print("\nPlot saved as lambda_sweep_pielm.png")
plt.show()

# Benchmark 1: Poisson 1D (PIELM)

**Equation:** `-u""(x) = f(x)`, x in [-1, 1], u(-1) = u(1) = 0

**Solution:** u(x) = sin(pi*x), f(x) = pi^2*sin(pi*x)

**PIELM training:** (W^2 * H) @ beta = f + BC rows. No u_true used in training.

In [ ]:
def solve_pinn_poisson(x, f_source, hidden=128, layers=1, iters=400, lr=0.005):
    x_grad = x.clone().detach().requires_grad_(True)
    net = [nn.Linear(1, hidden), nn.Tanh()]
    for _ in range(layers - 1):
        net += [nn.Linear(hidden, hidden), nn.Tanh()]
    net.append(nn.Linear(hidden, 1))
    model = nn.Sequential(*net).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bc_pts = torch.tensor([[-1.0], [1.0]], device=device)

    start = time.time()
    for _ in range(iters):
        opt.zero_grad()
        u = model(x_grad)
        u_x = torch.autograd.grad(u, x_grad, torch.ones_like(u), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x_grad, torch.ones_like(u_x), create_graph=True)[0]
        loss_pde = torch.mean((-u_xx - f_source) ** 2)
        loss_bc = model(bc_pts).pow(2).mean()
        loss = loss_pde + 10 * loss_bc
        loss.backward()
        opt.step()
    t_solve = time.time() - start
    with torch.no_grad():
        u_pred = model(x)
    return u_pred, t_solve

N_SEEDS = 20
h_dim = 4096
W_SCALE = 5.0

x = torch.linspace(-1, 1, 30000, device=device).view(-1, 1)
u_true = torch.sin(np.pi * x)  # ONLY for evaluation
f_source = (np.pi ** 2) * torch.sin(np.pi * x)

x_bc = torch.tensor([[-1.0], [1.0]], device=device)
u_bc = torch.zeros(2, 1, device=device)

print("=" * 70)
print("BENCHMARK 1: Poisson 1D - PIELM (no u_true in training)")
print("=" * 70)
print(f"w_scale={W_SCALE}, h_dim={h_dim}, seeds={N_SEEDS}")

data_rows = []
for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    u_pred_pl, _, _, _, ep_t = solve_pielm_poisson(
        x, f_source, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
    ep_err = (torch.norm(u_true - u_pred_pl) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Power Law)", "L2 Error": ep_err, "Time": ep_t})

    u_pred_g, _, _, _, eg_t = solve_pielm_poisson(
        x, f_source, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)
    eg_err = (torch.norm(u_true - u_pred_g) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Gaussian)", "L2 Error": eg_err, "Time": eg_t})

    torch.manual_seed(seed)
    u_pred_pinn, p_t = solve_pinn_poisson(x, f_source)
    p_err = (torch.norm(u_true - u_pred_pinn) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PINN (Light)", "L2 Error": p_err, "Time": p_t})

    if seed % 5 == 0:
        print(f"  Seed {seed}: PL={ep_err:.6f}, G={eg_err:.6f}, PINN={p_err:.4f}")

df = pd.DataFrame(data_rows)
summary = df.groupby("Method").agg({"L2 Error": ["mean", "std"], "Time": "mean"})
print("\n--- POISSON 1D RESULTS (PIELM) ---")
print(summary.to_markdown())

pl_err = summary.loc["PIELM (Power Law)", ("L2 Error", "mean")]
g_err = summary.loc["PIELM (Gaussian)", ("L2 Error", "mean")]
print(f"\nPower-Law is {g_err/pl_err:.1f}x more accurate than Gaussian")

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.rcParams.update({"font.size": 12, "font.family": "serif"})

x_plot = torch.linspace(-1, 1, 500, device=device).view(-1, 1)
u_plot_true = torch.sin(np.pi * x_plot)
f_plot = (np.pi ** 2) * torch.sin(np.pi * x_plot)

torch.manual_seed(0)
u_pl, _, _, _, _ = solve_pielm_poisson(x_plot, f_plot, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
torch.manual_seed(0)
u_g, _, _, _, _ = solve_pielm_poisson(x_plot, f_plot, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)

xnp = x_plot.cpu().numpy().flatten()
axes[0].plot(xnp, u_plot_true.cpu().numpy().flatten(), "k-", lw=2, label="True")
axes[0].plot(xnp, u_pl.cpu().detach().numpy().flatten(), "r--", lw=1.5, label="PIELM Power Law")
axes[0].plot(xnp, u_g.cpu().detach().numpy().flatten(), "g:", lw=1.5, label="PIELM Gaussian")
axes[0].set_title("(A) Poisson 1D: PIELM Solution")
axes[0].set_xlabel("x"); axes[0].set_ylabel("u(x)")
axes[0].legend(); axes[0].grid(True)

palette = {"PIELM (Power Law)": "#2ca02c", "PIELM (Gaussian)": "#ff7f0e", "PINN (Light)": "#1f77b4"}
sns.boxplot(x="Method", y="L2 Error", data=df, hue="Method", palette=palette, ax=axes[1],
            showmeans=True, meanprops={"marker":"o","markerfacecolor":"white",
            "markeredgecolor":"black","markersize":"8"}, legend=False)
axes[1].set_yscale("log")
axes[1].set_title("(B) L2 Error Distribution (20 seeds)")
axes[1].set_ylabel("L2 Relative Error")
axes[1].tick_params(axis="x", rotation=15)

err_pl = np.abs((u_plot_true - u_pl).cpu().detach().numpy().flatten())
err_g = np.abs((u_plot_true - u_g).cpu().detach().numpy().flatten())
axes[2].plot(xnp, err_pl, "r-", lw=1.5, label="|Error| Power Law")
axes[2].plot(xnp, err_g, "g-", lw=1.5, label="|Error| Gaussian")
axes[2].set_yscale("log")
axes[2].set_title("(C) Pointwise Error")
axes[2].set_xlabel("x"); axes[2].set_ylabel("|u_true - u_pred|")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("pielm_poisson1d.png", dpi=300, bbox_inches="tight")
print("\nPlot saved as pielm_poisson1d.png")
plt.show()

# Benchmark 2: Multi-Frequency Poisson 1D (PIELM)

**Equation:** `-u""(x) = f(x)`, x in [-1, 1], u(-1) = u(1) = 0

**Solution:** u(x) = sin(pi*x) + 0.3*sin(3*pi*x) + 0.1*sin(7*pi*x)

Key test: can PIELM with Power-Law weights capture multiple frequency modes better than Gaussian?

In [ ]:
def solve_pinn_multifreq(x, f_source, u_true_eval, hidden=256, layers=2, iters=5000, lr=0.001):
    x_grad = x.clone().detach().requires_grad_(True)
    net = [nn.Linear(1, hidden), nn.Tanh()]
    for _ in range(layers - 1):
        net += [nn.Linear(hidden, hidden), nn.Tanh()]
    net.append(nn.Linear(hidden, 1))
    model = nn.Sequential(*net).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=iters//3, gamma=0.5)
    bc_pts = torch.tensor([[-1.0], [1.0]], device=device)

    start = time.time()
    for _ in range(iters):
        opt.zero_grad()
        u = model(x_grad)
        u_x = torch.autograd.grad(u, x_grad, torch.ones_like(u), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x_grad, torch.ones_like(u_x), create_graph=True)[0]
        loss_pde = torch.mean((-u_xx - f_source)**2)
        loss_bc = model(bc_pts).pow(2).mean()
        loss = loss_pde + 10 * loss_bc
        loss.backward()
        opt.step()
        scheduler.step()
    t_solve = time.time() - start
    with torch.no_grad():
        err = (torch.norm(u_true_eval - model(x)) / torch.norm(u_true_eval)).item()
    return err, t_solve

N_SEEDS = 10
h_dim = 4096
W_SCALE = 30.0  # must cover up to 7*pi ~ 22

x = torch.linspace(-1, 1, 30000, device=device).view(-1, 1)
u_true = (torch.sin(np.pi * x)
          + 0.3 * torch.sin(3 * np.pi * x)
          + 0.1 * torch.sin(7 * np.pi * x))

f_source = (np.pi**2 * torch.sin(np.pi * x)
            + 0.3 * (3*np.pi)**2 * torch.sin(3 * np.pi * x)
            + 0.1 * (7*np.pi)**2 * torch.sin(7 * np.pi * x))

x_bc = torch.tensor([[-1.0], [1.0]], device=device)
u_bc = torch.zeros(2, 1, device=device)

print("=" * 70)
print("BENCHMARK 2: Multi-Frequency Poisson 1D - PIELM")
print("u(x) = sin(pi*x) + 0.3*sin(3*pi*x) + 0.1*sin(7*pi*x)")
print("=" * 70)
print(f"w_scale={W_SCALE}, h_dim={h_dim}")

data_rows = []
for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    u_pred_pl, _, _, _, ep_t = solve_pielm_poisson(
        x, f_source, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
    ep_err = (torch.norm(u_true - u_pred_pl) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Power Law)", "L2 Error": ep_err, "Time": ep_t})

    u_pred_g, _, _, _, eg_t = solve_pielm_poisson(
        x, f_source, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)
    eg_err = (torch.norm(u_true - u_pred_g) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Gaussian)", "L2 Error": eg_err, "Time": eg_t})

    torch.manual_seed(seed)
    p_err, p_t = solve_pinn_multifreq(x, f_source, u_true)
    data_rows.append({"Method": "PINN", "L2 Error": p_err, "Time": p_t})

    if seed % 5 == 0:
        print(f"  Seed {seed}: PL={ep_err:.6f}, G={eg_err:.6f}, PINN={p_err:.4f}")

df_mf = pd.DataFrame(data_rows)
summary_mf = df_mf.groupby("Method").agg({"L2 Error": ["mean", "std"], "Time": "mean"})
print("\n--- MULTI-FREQ POISSON RESULTS (PIELM) ---")
print(summary_mf.to_markdown())

pl_err = summary_mf.loc["PIELM (Power Law)", ("L2 Error", "mean")]
g_err = summary_mf.loc["PIELM (Gaussian)", ("L2 Error", "mean")]
print(f"\nPower-Law is {g_err/pl_err:.1f}x more accurate than Gaussian")

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x_plot = torch.linspace(-1, 1, 500, device=device).view(-1, 1)
u_plot_true = (torch.sin(np.pi * x_plot) + 0.3 * torch.sin(3 * np.pi * x_plot)
               + 0.1 * torch.sin(7 * np.pi * x_plot))
f_plot = (np.pi**2 * torch.sin(np.pi * x_plot)
          + 0.3 * (3*np.pi)**2 * torch.sin(3 * np.pi * x_plot)
          + 0.1 * (7*np.pi)**2 * torch.sin(7 * np.pi * x_plot))

torch.manual_seed(0)
u_pl, _, _, _, _ = solve_pielm_poisson(x_plot, f_plot, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
torch.manual_seed(0)
u_g, _, _, _, _ = solve_pielm_poisson(x_plot, f_plot, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)

xnp = x_plot.cpu().numpy().flatten()
axes[0].plot(xnp, u_plot_true.cpu().numpy().flatten(), "k-", lw=2, label="True")
axes[0].plot(xnp, u_pl.cpu().detach().numpy().flatten(), "r--", lw=1.5, label="PIELM Power Law")
axes[0].plot(xnp, u_g.cpu().detach().numpy().flatten(), "g:", lw=1.5, label="PIELM Gaussian")
axes[0].set_title("(A) Multi-Freq Poisson: PIELM"); axes[0].set_xlabel("x"); axes[0].set_ylabel("u(x)")
axes[0].legend(); axes[0].grid(True)

palette = {"PIELM (Power Law)": "#2ca02c", "PIELM (Gaussian)": "#ff7f0e", "PINN": "#1f77b4"}
sns.boxplot(x="Method", y="L2 Error", data=df_mf, hue="Method", palette=palette, ax=axes[1],
            showmeans=True, meanprops={"marker":"o","markerfacecolor":"white",
            "markeredgecolor":"black","markersize":"8"}, legend=False)
axes[1].set_yscale("log"); axes[1].set_title("(B) L2 Error (10 seeds)")
axes[1].set_ylabel("L2 Relative Error"); axes[1].tick_params(axis="x", rotation=15)

err_pl = np.abs((u_plot_true - u_pl).cpu().detach().numpy().flatten())
err_g = np.abs((u_plot_true - u_g).cpu().detach().numpy().flatten())
axes[2].plot(xnp, err_pl, "r-", lw=1.5, label="|Error| Power Law")
axes[2].plot(xnp, err_g, "g-", lw=1.5, label="|Error| Gaussian")
axes[2].set_yscale("log"); axes[2].set_title("(C) Pointwise Error")
axes[2].set_xlabel("x"); axes[2].set_ylabel("|u_true - u_pred|")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("pielm_multifreq_poisson.png", dpi=300, bbox_inches="tight")
print("\nPlot saved as pielm_multifreq_poisson.png")
plt.show()

# Benchmark 3: Helmholtz 1D (PIELM)

**Equation:** `-u""(x) - k^2*u(x) = f(x)`, x in [-1, 1], u(-1) = u(1) = 0

**Solution:** u(x) = sin(pi*x)*exp(-4*x^2), k = 10

**PIELM:** H_pde = (W^2 - k^2) * H

In [ ]:
def solve_pinn_helmholtz(x, f_source, k, u_true_eval, hidden=256, layers=2, iters=5000, lr=0.001):
    x_grad = x.clone().detach().requires_grad_(True)
    net = [nn.Linear(1, hidden), nn.Tanh()]
    for _ in range(layers - 1):
        net += [nn.Linear(hidden, hidden), nn.Tanh()]
    net.append(nn.Linear(hidden, 1))
    model = nn.Sequential(*net).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=iters//3, gamma=0.5)
    bc_pts = torch.tensor([[-1.0], [1.0]], device=device)

    start = time.time()
    for _ in range(iters):
        opt.zero_grad()
        u = model(x_grad)
        u_x = torch.autograd.grad(u, x_grad, torch.ones_like(u), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x_grad, torch.ones_like(u_x), create_graph=True)[0]
        residual = -u_xx - k**2 * u - f_source
        loss_pde = torch.mean(residual**2)
        loss_bc = model(bc_pts).pow(2).mean()
        loss = loss_pde + 10 * loss_bc
        loss.backward()
        opt.step()
        scheduler.step()
    t_solve = time.time() - start
    with torch.no_grad():
        err = (torch.norm(u_true_eval - model(x)) / torch.norm(u_true_eval)).item()
    return err, t_solve

N_SEEDS = 10
h_dim = 4096
k = 10.0
W_SCALE = 15.0

x = torch.linspace(-1, 1, 30000, device=device).view(-1, 1)
envelope = torch.exp(-4 * x**2)
u_true = torch.sin(np.pi * x) * envelope

u_xx = ((-np.pi**2 + 64 * x**2 - 8) * torch.sin(np.pi * x)
        - 16 * np.pi * x * torch.cos(np.pi * x)) * envelope
f_source = -u_xx - k**2 * u_true

x_bc = torch.tensor([[-1.0], [1.0]], device=device)
u_bc = torch.zeros(2, 1, device=device)

print(f"BC check: u(-1) = {u_true[0].item():.2e}, u(1) = {u_true[-1].item():.2e}")
print("=" * 70)
print(f"BENCHMARK 3: Helmholtz 1D (k={k}) - PIELM")
print("=" * 70)
print(f"w_scale={W_SCALE}, h_dim={h_dim}")

data_rows = []
for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    u_pred_pl, _, _, _, ep_t = solve_pielm_helmholtz(
        x, f_source, k, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
    ep_err = (torch.norm(u_true - u_pred_pl) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Power Law)", "L2 Error": ep_err, "Time": ep_t})

    u_pred_g, _, _, _, eg_t = solve_pielm_helmholtz(
        x, f_source, k, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)
    eg_err = (torch.norm(u_true - u_pred_g) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Gaussian)", "L2 Error": eg_err, "Time": eg_t})

    torch.manual_seed(seed)
    p_err, p_t = solve_pinn_helmholtz(x, f_source, k, u_true)
    data_rows.append({"Method": "PINN", "L2 Error": p_err, "Time": p_t})

    if seed % 5 == 0:
        print(f"  Seed {seed}: PL={ep_err:.6f}, G={eg_err:.6f}, PINN={p_err:.4f}")

df_hz = pd.DataFrame(data_rows)
summary_hz = df_hz.groupby("Method").agg({"L2 Error": ["mean", "std"], "Time": "mean"})
print("\n--- HELMHOLTZ RESULTS (PIELM) ---")
print(summary_hz.to_markdown())

pl_err = summary_hz.loc["PIELM (Power Law)", ("L2 Error", "mean")]
g_err = summary_hz.loc["PIELM (Gaussian)", ("L2 Error", "mean")]
print(f"\nPower-Law is {g_err/pl_err:.1f}x more accurate than Gaussian")

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x_plot = torch.linspace(-1, 1, 500, device=device).view(-1, 1)
env_plot = torch.exp(-4 * x_plot**2)
u_plot_true = torch.sin(np.pi * x_plot) * env_plot
u_xx_plot = ((-np.pi**2 + 64*x_plot**2 - 8)*torch.sin(np.pi*x_plot)
             - 16*np.pi*x_plot*torch.cos(np.pi*x_plot)) * env_plot
f_plot = -u_xx_plot - k**2 * u_plot_true

torch.manual_seed(0)
u_pl, _, _, _, _ = solve_pielm_helmholtz(x_plot, f_plot, k, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
torch.manual_seed(0)
u_g, _, _, _, _ = solve_pielm_helmholtz(x_plot, f_plot, k, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)

xnp = x_plot.cpu().numpy().flatten()
axes[0].plot(xnp, u_plot_true.cpu().numpy().flatten(), "k-", lw=2, label="True")
axes[0].plot(xnp, u_pl.cpu().detach().numpy().flatten(), "r--", lw=1.5, label="PIELM Power Law")
axes[0].plot(xnp, u_g.cpu().detach().numpy().flatten(), "g:", lw=1.5, label="PIELM Gaussian")
axes[0].set_title(f"(A) Helmholtz (k={k}): PIELM"); axes[0].set_xlabel("x"); axes[0].set_ylabel("u(x)")
axes[0].legend(); axes[0].grid(True)

palette = {"PIELM (Power Law)": "#2ca02c", "PIELM (Gaussian)": "#ff7f0e", "PINN": "#1f77b4"}
sns.boxplot(x="Method", y="L2 Error", data=df_hz, hue="Method", palette=palette, ax=axes[1],
            showmeans=True, meanprops={"marker":"o","markerfacecolor":"white",
            "markeredgecolor":"black","markersize":"8"}, legend=False)
axes[1].set_yscale("log"); axes[1].set_title("(B) L2 Error (10 seeds)")
axes[1].set_ylabel("L2 Relative Error"); axes[1].tick_params(axis="x", rotation=15)

err_pl = np.abs((u_plot_true - u_pl).cpu().detach().numpy().flatten())
err_g = np.abs((u_plot_true - u_g).cpu().detach().numpy().flatten())
axes[2].plot(xnp, err_pl, "r-", lw=1.5, label="|Error| Power Law")
axes[2].plot(xnp, err_g, "g-", lw=1.5, label="|Error| Gaussian")
axes[2].set_yscale("log"); axes[2].set_title("(C) Pointwise Error")
axes[2].set_xlabel("x"); axes[2].set_ylabel("|u_true - u_pred|")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig("pielm_helmholtz.png", dpi=300, bbox_inches="tight")
print("\nPlot saved as pielm_helmholtz.png")
plt.show()

# Benchmark 4: High-Frequency Poisson 1D (PIELM)

**Equation:** `-u""(x) = f(x)`, x in [-1, 1], u(-1) = u(1) = 0

**Solution:** u(x) = sin(10*pi*x) -- hardest frequency test (10*pi ~ 31.4)

In [ ]:
N_SEEDS = 10
h_dim = 4096
W_SCALE = 40.0

x = torch.linspace(-1, 1, 30000, device=device).view(-1, 1)
u_true = torch.sin(10 * np.pi * x)
f_source = (10 * np.pi)**2 * torch.sin(10 * np.pi * x)

x_bc = torch.tensor([[-1.0], [1.0]], device=device)
u_bc = torch.zeros(2, 1, device=device)

print("=" * 70)
print("BENCHMARK 4: High-Frequency Poisson 1D - PIELM")
print("u(x) = sin(10*pi*x),  freq = 10*pi ~ 31.4")
print("=" * 70)
print(f"w_scale={W_SCALE}, h_dim={h_dim}")

data_rows = []
for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    u_pred_pl, _, _, _, ep_t = solve_pielm_poisson(
        x, f_source, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
    ep_err = (torch.norm(u_true - u_pred_pl) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Power Law)", "L2 Error": ep_err, "Time": ep_t})

    u_pred_g, _, _, _, eg_t = solve_pielm_poisson(
        x, f_source, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)
    eg_err = (torch.norm(u_true - u_pred_g) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Gaussian)", "L2 Error": eg_err, "Time": eg_t})

    if seed % 5 == 0:
        print(f"  Seed {seed}: PL={ep_err:.6f}, G={eg_err:.6f}")

df_hf = pd.DataFrame(data_rows)
summary_hf = df_hf.groupby("Method").agg({"L2 Error": ["mean", "std"], "Time": "mean"})
print("\n--- HIGH-FREQ POISSON RESULTS (PIELM) ---")
print(summary_hf.to_markdown())

pl_err = summary_hf.loc["PIELM (Power Law)", ("L2 Error", "mean")]
g_err = summary_hf.loc["PIELM (Gaussian)", ("L2 Error", "mean")]
print(f"\nPower-Law is {g_err/pl_err:.1f}x more accurate than Gaussian")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_plot = torch.linspace(-1, 1, 500, device=device).view(-1, 1)
u_plot_true = torch.sin(10 * np.pi * x_plot)
f_plot = (10 * np.pi)**2 * torch.sin(10 * np.pi * x_plot)

torch.manual_seed(0)
u_pl, _, _, _, _ = solve_pielm_poisson(x_plot, f_plot, x_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
torch.manual_seed(0)
u_g, _, _, _, _ = solve_pielm_poisson(x_plot, f_plot, x_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)

xnp = x_plot.cpu().numpy().flatten()
axes[0].plot(xnp, u_plot_true.cpu().numpy().flatten(), "k-", lw=2, label="True")
axes[0].plot(xnp, u_pl.cpu().detach().numpy().flatten(), "r--", lw=1.5, label="PIELM Power Law")
axes[0].plot(xnp, u_g.cpu().detach().numpy().flatten(), "g:", lw=1.5, label="PIELM Gaussian")
axes[0].set_title("(A) High-Freq Poisson: PIELM"); axes[0].set_xlabel("x"); axes[0].set_ylabel("u(x)")
axes[0].legend(); axes[0].grid(True)

palette = {"PIELM (Power Law)": "#2ca02c", "PIELM (Gaussian)": "#ff7f0e"}
sns.boxplot(x="Method", y="L2 Error", data=df_hf, hue="Method", palette=palette, ax=axes[1],
            showmeans=True, meanprops={"marker":"o","markerfacecolor":"white",
            "markeredgecolor":"black","markersize":"8"}, legend=False)
axes[1].set_yscale("log"); axes[1].set_title("(B) L2 Error (10 seeds)")
axes[1].set_ylabel("L2 Relative Error"); axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("pielm_highfreq_poisson.png", dpi=300, bbox_inches="tight")
print("\nPlot saved as pielm_highfreq_poisson.png")
plt.show()

# Benchmark 5: Poisson 2D (PIELM)

**Equation:** `-laplacian(u) = f(x,y)`, (x,y) in [-1,1]^2, u = 0 on boundary

**Solution:** u(x,y) = sin(pi*x)*sin(2*pi*y)

**PIELM:** H_pde[i,j] = (w1_j^2 + w2_j^2) * H[i,j]

In [ ]:
N_SEEDS = 10
h_dim = 4096
W_SCALE = 10.0

n_int = 150
x1d = torch.linspace(-1, 1, n_int, device=device)
xx, yy = torch.meshgrid(x1d, x1d, indexing="ij")
X_int = torch.stack([xx.flatten(), yy.flatten()], dim=1)

u_true_int = (torch.sin(np.pi * X_int[:, 0:1]) *
              torch.sin(2 * np.pi * X_int[:, 1:2]))
f_int = (np.pi**2 + (2*np.pi)**2) * u_true_int

n_bc_edge = 100
bc_points = []
for v in torch.linspace(-1, 1, n_bc_edge, device=device):
    bc_points.append([v.item(), -1.0])
    bc_points.append([v.item(),  1.0])
    bc_points.append([-1.0, v.item()])
    bc_points.append([ 1.0, v.item()])
X_bc = torch.tensor(bc_points, device=device).unique(dim=0)
u_bc = torch.zeros(X_bc.shape[0], 1, device=device)

print("=" * 70)
print("BENCHMARK 5: Poisson 2D - PIELM")
print("u(x,y) = sin(pi*x) * sin(2*pi*y)")
print("=" * 70)
print(f"Interior: {X_int.shape[0]}, Boundary: {X_bc.shape[0]}, h_dim={h_dim}")

data_rows = []
for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    u_pred_pl, _, _, _, ep_t = solve_pielm_poisson_2d(
        X_int, f_int, X_bc, u_bc, h_dim, "power", w_scale=W_SCALE)
    ep_err = (torch.norm(u_true_int - u_pred_pl) / torch.norm(u_true_int)).item()
    data_rows.append({"Method": "PIELM (Power Law)", "L2 Error": ep_err, "Time": ep_t})

    u_pred_g, _, _, _, eg_t = solve_pielm_poisson_2d(
        X_int, f_int, X_bc, u_bc, h_dim, "gaussian", w_scale=W_SCALE)
    eg_err = (torch.norm(u_true_int - u_pred_g) / torch.norm(u_true_int)).item()
    data_rows.append({"Method": "PIELM (Gaussian)", "L2 Error": eg_err, "Time": eg_t})

    if seed % 5 == 0:
        print(f"  Seed {seed}: PL={ep_err:.6f}, G={eg_err:.6f}")

df_2d = pd.DataFrame(data_rows)
summary_2d = df_2d.groupby("Method").agg({"L2 Error": ["mean", "std"], "Time": "mean"})
print("\n--- POISSON 2D RESULTS (PIELM) ---")
print(summary_2d.to_markdown())

pl_err = summary_2d.loc["PIELM (Power Law)", ("L2 Error", "mean")]
g_err = summary_2d.loc["PIELM (Gaussian)", ("L2 Error", "mean")]
print(f"\nPower-Law is {g_err/pl_err:.1f}x more accurate than Gaussian")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
torch.manual_seed(0)
u_pl_2d, _, _, _, _ = solve_pielm_poisson_2d(
    X_int, f_int, X_bc, u_bc, h_dim, "power", w_scale=W_SCALE)

n = n_int
u_true_grid = u_true_int.cpu().numpy().reshape(n, n)
u_pred_grid = u_pl_2d.cpu().detach().numpy().reshape(n, n)
err_grid = np.abs(u_true_grid - u_pred_grid)

xnp = x1d.cpu().numpy()
im0 = axes[0].pcolormesh(xnp, xnp, u_true_grid, shading="auto", cmap="RdBu_r")
axes[0].set_title("(A) True Solution"); plt.colorbar(im0, ax=axes[0])
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")

im1 = axes[1].pcolormesh(xnp, xnp, u_pred_grid, shading="auto", cmap="RdBu_r")
axes[1].set_title("(B) PIELM (Power Law)"); plt.colorbar(im1, ax=axes[1])
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")

im2 = axes[2].pcolormesh(xnp, xnp, err_grid, shading="auto", cmap="hot_r")
axes[2].set_title("(C) Pointwise |Error|"); plt.colorbar(im2, ax=axes[2])
axes[2].set_xlabel("x"); axes[2].set_ylabel("y")

plt.tight_layout()
plt.savefig("pielm_poisson2d.png", dpi=300, bbox_inches="tight")
print("\nPlot saved as pielm_poisson2d.png")
plt.show()

# Benchmark 6: Advection-Diffusion 1D (PIELM)

**Equation:** `-eps*u""(x) + u'(x) = 0`, x in [0, 1], u(0) = 0, u(1) = 1

**Exact:** u(x) = (exp(x/eps) - 1) / (exp(1/eps) - 1), eps = 0.02

**PIELM:** H_pde = eps*W^2*sin(...) + W*cos(...)

In [ ]:
N_SEEDS = 10
h_dim = 4096
eps = 0.02
W_SCALE = 60.0

x = torch.linspace(0, 1, 30000, device=device).view(-1, 1)
u_true = (torch.exp(x / eps) - 1) / (torch.exp(torch.tensor(1.0 / eps, device=device)) - 1)

x_bc = torch.tensor([[0.0], [1.0]], device=device)
u_bc = torch.tensor([[0.0], [1.0]], device=device)

print("=" * 70)
print(f"BENCHMARK 6: Advection-Diffusion 1D (eps={eps}) - PIELM")
print("=" * 70)
print(f"w_scale={W_SCALE}, h_dim={h_dim}")

data_rows = []
for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    u_pred_pl, _, _, _, ep_t = solve_pielm_advdiff(
        x, x_bc, u_bc, h_dim, eps, "power", w_scale=W_SCALE)
    ep_err = (torch.norm(u_true - u_pred_pl) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Power Law)", "L2 Error": ep_err, "Time": ep_t})

    u_pred_g, _, _, _, eg_t = solve_pielm_advdiff(
        x, x_bc, u_bc, h_dim, eps, "gaussian", w_scale=W_SCALE)
    eg_err = (torch.norm(u_true - u_pred_g) / torch.norm(u_true)).item()
    data_rows.append({"Method": "PIELM (Gaussian)", "L2 Error": eg_err, "Time": eg_t})

    if seed % 5 == 0:
        print(f"  Seed {seed}: PL={ep_err:.6f}, G={eg_err:.6f}")

df_ad = pd.DataFrame(data_rows)
summary_ad = df_ad.groupby("Method").agg({"L2 Error": ["mean", "std"], "Time": "mean"})
print("\n--- ADVECTION-DIFFUSION RESULTS (PIELM) ---")
print(summary_ad.to_markdown())

pl_err = summary_ad.loc["PIELM (Power Law)", ("L2 Error", "mean")]
g_err = summary_ad.loc["PIELM (Gaussian)", ("L2 Error", "mean")]
print(f"\nPower-Law is {g_err/pl_err:.1f}x more accurate than Gaussian")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_plot = torch.linspace(0, 1, 500, device=device).view(-1, 1)
u_plot_true = (torch.exp(x_plot / eps) - 1) / (torch.exp(torch.tensor(1.0/eps, device=device)) - 1)

torch.manual_seed(0)
u_pl, _, _, _, _ = solve_pielm_advdiff(x_plot, x_bc, u_bc, h_dim, eps, "power", w_scale=W_SCALE)
torch.manual_seed(0)
u_g, _, _, _, _ = solve_pielm_advdiff(x_plot, x_bc, u_bc, h_dim, eps, "gaussian", w_scale=W_SCALE)

xnp = x_plot.cpu().numpy().flatten()
axes[0].plot(xnp, u_plot_true.cpu().numpy().flatten(), "k-", lw=2, label="True")
axes[0].plot(xnp, u_pl.cpu().detach().numpy().flatten(), "r--", lw=1.5, label="PIELM Power Law")
axes[0].plot(xnp, u_g.cpu().detach().numpy().flatten(), "g:", lw=1.5, label="PIELM Gaussian")
axes[0].set_title(f"(A) Advection-Diffusion (eps={eps}): PIELM")
axes[0].set_xlabel("x"); axes[0].set_ylabel("u(x)")
axes[0].legend(); axes[0].grid(True)

palette = {"PIELM (Power Law)": "#2ca02c", "PIELM (Gaussian)": "#ff7f0e"}
sns.boxplot(x="Method", y="L2 Error", data=df_ad, hue="Method", palette=palette, ax=axes[1],
            showmeans=True, meanprops={"marker":"o","markerfacecolor":"white",
            "markeredgecolor":"black","markersize":"8"}, legend=False)
axes[1].set_yscale("log"); axes[1].set_title("(B) L2 Error (10 seeds)")
axes[1].set_ylabel("L2 Relative Error"); axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("pielm_advdiff.png", dpi=300, bbox_inches="tight")
print("\nPlot saved as pielm_advdiff.png")
plt.show()

# Benchmark 7: Burgers Equation (Supervised ELM -- NOT PIELM)

**Equation:** u_t + u*u_x = nu*u_xx (nonlinear)

**Why not PIELM?** The u*u_x term is **nonlinear in beta** -- the PDE operator cannot be factored as a linear system in beta. PIELM only works for linear PDEs.

This benchmark keeps supervised ELM for both methods. The paper should explicitly acknowledge this distinction.

In [ ]:
from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

nu = 0.01 / np.pi

def burgers_reference(nx=1024, nu=0.01/np.pi):
    x = np.linspace(-1, 1, nx)
    dx = x[1] - x[0]
    u0 = -np.sin(np.pi * x)
    def rhs(t, u):
        dudt = np.zeros_like(u)
        u_x = np.zeros_like(u)
        u_x[1:-1] = (u[2:] - u[:-2]) / (2 * dx)
        u_xx = np.zeros_like(u)
        u_xx[1:-1] = (u[2:] - 2*u[1:-1] + u[:-2]) / dx**2
        dudt[1:-1] = -u[1:-1] * u_x[1:-1] + nu * u_xx[1:-1]
        dudt[0] = 0; dudt[-1] = 0
        return dudt
    t_eval = np.linspace(0, 1, 201)
    sol = solve_ivp(rhs, [0, 1], u0, t_eval=t_eval, method="RK45",
                    rtol=1e-8, atol=1e-10, max_step=dx/2)
    print(f"Reference solver: nx={nx}, status={sol.message}")
    return x, t_eval, sol.y

print("Computing Burgers reference solution...")
x_ref, t_ref, u_ref = burgers_reference(nx=1024)

def solve_elm_supervised(X, u_true, h_dim, init_type="power", w_scale=3.0, b_scale=1.0, lambd=1e-6):
    input_dim = X.shape[1]
    W = power_law_init((input_dim, h_dim), w_scale=w_scale) if init_type == "power" \
        else gaussian_init((input_dim, h_dim), w_scale=w_scale)
    b = torch.randn(h_dim, device=device) * b_scale
    start = time.time()
    H = torch.sin(X @ W + b)
    HTH = H.t() @ H + lambd * torch.eye(h_dim, device=device)
    beta = torch.linalg.solve(HTH, H.t() @ u_true)
    t_solve = time.time() - start
    u_pred = H @ beta
    err = (torch.norm(u_true - u_pred) / torch.norm(u_true)).item()
    return u_pred, err, t_solve, W, b, beta

def solve_pinn_burgers(X_train, u_true, layers=2, hidden=128, adam_iters=5000, lbfgs_iters=500, lr=0.001):
    X_res = X_train.clone().detach().requires_grad_(True)
    net = [nn.Linear(2, hidden), nn.Tanh()]
    for _ in range(layers-1):
        net += [nn.Linear(hidden, hidden), nn.Tanh()]
    net.append(nn.Linear(hidden, 1))
    model = nn.Sequential(*net).to(device)
    mask = (X_train[:,0] == -1) | (X_train[:,0] == 1) | (X_train[:,1] == 0)
    X_bc = X_train[mask]; u_bc = u_true[mask]

    def compute_loss():
        u = model(X_res)
        grads = torch.autograd.grad(u, X_res, torch.ones_like(u), create_graph=True)[0]
        u_x, u_t = grads[:, 0:1], grads[:, 1:2]
        u_xx = torch.autograd.grad(u_x, X_res, torch.ones_like(u_x), create_graph=True)[0][:, 0:1]
        loss_pde = torch.mean((u_t + u*u_x - nu*u_xx)**2)
        loss_bc = torch.mean((model(X_bc) - u_bc)**2)
        return loss_pde + 25 * loss_bc

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=adam_iters//2, gamma=0.5)
    start = time.time()
    for _ in range(adam_iters):
        opt.zero_grad()
        loss = compute_loss()
        loss.backward()
        opt.step()
        scheduler.step()
    if lbfgs_iters > 0:
        opt_lbfgs = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=20,
                                       history_size=50, line_search_fn="strong_wolfe")
        def closure():
            opt_lbfgs.zero_grad()
            loss = compute_loss()
            loss.backward()
            return loss
        for _ in range(lbfgs_iters):
            opt_lbfgs.step(closure)
    t_total = time.time() - start
    with torch.no_grad():
        err = (torch.norm(u_true - model(X_train)) / torch.norm(u_true)).item()
    return err, t_total

nx_train, nt_train = 150, 150
x_b = np.linspace(-1, 1, nx_train)
t_b = np.linspace(0, 1, nt_train)
interp = RegularGridInterpolator((x_ref, t_ref), u_ref, method="cubic")
X_grid_np, T_grid_np = np.meshgrid(x_b, t_b, indexing="ij")
pts = np.stack([X_grid_np.flatten(), T_grid_np.flatten()], axis=1)
u_ref_on_grid = interp(pts).reshape(-1, 1)

X_burgers = torch.tensor(pts, dtype=torch.float32, device=device)
u_b_true = torch.tensor(u_ref_on_grid, dtype=torch.float32, device=device)

W_SCALE = 3.0; B_SCALE = 1.0; H_DIM = 8192; LAMBDA = 1e-6
N_SEEDS = 5

print(f"Training grid: {nx_train}x{nt_train} = {X_burgers.shape[0]} points")
print("=" * 70)
print("BENCHMARK 7: Burgers (Supervised - nonlinear, PIELM not applicable)")
print("=" * 70)

all_results = []
for seed in range(N_SEEDS):
    torch.manual_seed(seed); np.random.seed(seed)
    print(f"  Seed {seed+1}/{N_SEEDS}...")

    _, ep_err, ep_t, _, _, _ = solve_elm_supervised(
        X_burgers, u_b_true, H_DIM, "power", w_scale=W_SCALE, b_scale=B_SCALE, lambd=LAMBDA)
    all_results.append({"Method": "ELM (Power)", "L2 Error": ep_err, "Time": ep_t})

    _, eg_err, eg_t, _, _, _ = solve_elm_supervised(
        X_burgers, u_b_true, H_DIM, "gaussian", w_scale=W_SCALE, b_scale=B_SCALE, lambd=LAMBDA)
    all_results.append({"Method": "ELM (Gauss)", "L2 Error": eg_err, "Time": eg_t})

    torch.manual_seed(seed)
    lp_err, lp_t = solve_pinn_burgers(X_burgers, u_b_true, layers=1, adam_iters=5000, lbfgs_iters=500)
    all_results.append({"Method": "PINN (Light)", "L2 Error": lp_err, "Time": lp_t})

    torch.manual_seed(seed)
    dp_err, dp_t = solve_pinn_burgers(X_burgers, u_b_true, layers=4, adam_iters=10000, lbfgs_iters=1000)
    all_results.append({"Method": "PINN (Deep)", "L2 Error": dp_err, "Time": dp_t})

df_b = pd.DataFrame(all_results)
summary_b = df_b.groupby("Method").agg({"L2 Error": ["mean", "std"], "Time": "mean"})
print("\n--- BURGERS (Supervised -- both ELM and PINN use reference) ---")
print(summary_b.to_markdown())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x="Method", y="L2 Error", data=df_b, ax=axes[0], palette="Set2",
            showmeans=True, meanprops={"marker":"o","markerfacecolor":"white",
            "markeredgecolor":"black","markersize":"8"})
axes[0].set_yscale("log"); axes[0].set_title("L2 Error Distribution")
axes[0].set_ylabel("L2 Relative Error"); axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=df_b, x="Method", y="Time", ax=axes[1], palette="plasma")
axes[1].set_title("Training Time"); axes[1].set_ylabel("Time (s)")
axes[1].tick_params(axis="x", rotation=15); axes[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.savefig("pielm_burgers_supervised.png", dpi=300, bbox_inches="tight")
print("\nPlot saved as pielm_burgers_supervised.png")
plt.show()

# Summary

| Benchmark | Method | Training uses u_true? | Notes |
|---|---|---|---|
| Poisson 1D | **PIELM** | No | H_pde = W^2 * H |
| Multi-Freq Poisson | **PIELM** | No | Same operator, harder target |
| Helmholtz 1D | **PIELM** | No | H_pde = (W^2-k^2) * H |
| High-Freq Poisson | **PIELM** | No | u = sin(10*pi*x), stress test |
| Poisson 2D | **PIELM** | No | H_pde = (W1^2+W2^2) * H |
| Advection-Diffusion | **PIELM** | No | H_pde = eps*W^2*H + W*cos(...) |
| Burgers | Supervised ELM | **Yes** | Nonlinear -- PIELM not applicable |

**Key improvements over previous notebook:**
1. ELM and PINN now solve the **same problem** for all linear PDEs (fair comparison)
2. Weight scale w_scale is tuned per problem (not fixed at 0.1)
3. u_true only used for evaluation, never in PIELM training
4. Burgers explicitly marked as supervised with justification

**Reference:** Dwivedi & Srinivasan (2020), Physics Informed Extreme Learning Machine, arXiv:1907.03507